# Tidy Data와 연구 시각화

## OWID CO₂ 자료의 구조 변환과 집계

같은 값을 long과 wide 구조로 바꾸고, `groupby`가 행을 그룹으로 나눈 뒤 어떻게 요약하는지 확인합니다. 연구 질문에 맞는 집계함수와 그림을 선택하고, 변환 전후의 관측 단위와 값이 의도대로 보존됐는지 검증합니다.


## 이번 교시의 학습목표

수업이 끝나면 다음을 수행할 수 있습니다.

- 변수, 관측, 관측단위를 구분하고 현재 표의 `grain`과 `key`를 설명합니다.
- `groupby`의 분할–적용–결합 과정을 설명하고, 평균·중앙값·합계·개수가 답하는 질문을 구분합니다.
- `pivot`, `melt`, `groupby`, `agg`로 표를 변형한 뒤 행의 의미, key와 값의 보존 여부를 검증합니다.
- 시간 변화, 그룹 비교, 분포 비교에 맞는 시각화를 선택하고 축·단위·집계함수·표본 범위를 함께 제시합니다.


## 이번 교시에서 사용할 데이터

### 강의 시연: OWID CO₂ 국가–연도 표

- **규모:** 175행, 5개 지역, 1990~2024년
- **현재 grain:** 지역 × 연도
- **reshape에 사용할 key:** `code + year`
- **measure:** 총 CO₂ 배출량 `emissions_total`

원자료는 국가가 행에 반복되는 long 구조입니다. `pivot`을 적용하면 연도가 행, 국가가 열인 wide 구조가 됩니다. 구조가 달라져도 같은 `code–year`의 값이 보존되는지 확인합니다.

### 적용 실습: UCI Bike Sharing 시간별 자료

- **규모:** 17,379행, 2011~2012년
- **현재 grain:** 날짜 × 시간
- **집계에 사용할 열:** `hr`, `workingday`, `weekday`
- **measure:** 시간별 총 대여건수 `cnt`

시간대별 평균을 계산하면 grain이 날짜×시간에서 시간 또는 시간×근무일로 바뀝니다. `groupby` 뒤의 한 행이 무엇을 뜻하는지, 평균과 합계 가운데 어떤 집계가 질문에 맞는지 먼저 정합니다. 정규화된 기온·습도 값은 원래 물리 단위와 다르므로 축 label에 단위를 임의로 붙이지 않습니다.


## Tidy Data가 분석 절차를 단순하게 만드는 이유

자료마다 변수와 관측의 위치가 다르면 변형·집계·시각화 코드를 매번 새로 작성해야 합니다. Wickham (2014)은 반복 가능한 분석을 위해 다음 구조를 제안했습니다.

1. 각 **변수**는 하나의 열을 이룹니다.
2. 각 **관측**은 하나의 행을 이룹니다.
3. 각 **관측단위**는 하나의 표를 이룹니다.

이 원칙은 특정 모양을 무조건 선택하라는 규칙이 아닙니다. 분석 함수가 요구하는 입력 구조와 연구 질문을 연결하는 공통 언어입니다.

출처: [Wickham (2014), *Tidy Data*](https://doi.org/10.18637/jss.v059.i10)


## long과 wide: 같은 관측값을 배치하는 두 방식

**long 형식**에서는 국가가 한 열의 값으로 반복되고, **wide 형식**에서는 국가가 여러 열 이름으로 펼쳐집니다.

| 목적 | 더 자연스러운 형식의 예 |
|---|---|
| 국가를 색으로 나눈 선 그래프 | long |
| 같은 연도의 국가값을 열별로 비교 | wide |
| 변수별 통계모형 입력 | 대체로 long |
| 사람이 읽는 교차표 | 대체로 wide |

`pivot`은 행의 값을 열 이름으로 펼치고, `melt`는 여러 열 이름을 다시 한 열의 값으로 모읍니다. 다음 실행 셀에서는 동일한 여섯 관측을 `long → wide → long`으로 바꾸고 값이 보존됐는지 확인합니다.


In [ ]:
import pandas as pd
from IPython.display import display

long_df = pd.DataFrame(
    {
        "year": [2020, 2020, 2021, 2021, 2022, 2022],
        "code": ["KOR", "JPN"] * 3,
        "co2_mt": [650, 1080, 625, 1065, 602, 1055],
    }
)
wide_df = long_df.pivot(index="year", columns="code", values="co2_mt").reset_index()
wide_df.columns.name = None
long_again = (
    wide_df.melt(id_vars="year", var_name="code", value_name="co2_mt")
    .sort_values(["year", "code"])
    .reset_index(drop=True)
)
expected = long_df.sort_values(["year", "code"]).reset_index(drop=True)

print("[long] 한 행 = 한 국가의 한 연도")
print("국가가 code 열의 값으로 반복됩니다.")
display(long_df)
print("\n[wide] 한 행 = 한 연도")
print("국가가 KOR·JPN 열로 펼쳐집니다.")
display(wide_df)
print("\n[wide에서 다시 만든 long]")
display(long_again)
print(f"key별 값 보존: {expected.equals(long_again)}")
print(f"CO₂ 관측값 수 보존: {len(long_df)} → {int(wide_df.drop(columns='year').notna().sum().sum())}")
print(f"shape 변화: long {long_df.shape} → wide {wide_df.shape} → long {long_again.shape}")


## reshape와 aggregation은 결과의 의미가 다르다

| 작업 | 대표 함수 | 값에 생기는 일 | grain에 생기는 일 |
|---|---|---|---|
| reshape | `pivot`, `melt` | 같은 값의 위치가 바뀜 | 원래 관측 단위를 유지할 수 있음 |
| aggregation | `groupby`, `agg` | 평균·합계·개수 등 새 값을 계산 | 여러 관측이 하나의 요약 행으로 합쳐짐 |

날짜×시간 대여자료를 `시간대×근무일` 평균으로 묶으면 결과 한 행은 특정 날짜가 아니라 여러 날짜의 요약입니다. 분석 결과를 보고할 때는 사용한 집계함수와 집계 후 grain을 함께 적습니다.


## groupby는 나누기·계산하기·합치기를 수행한다

`groupby`는 표를 조건별로 나눈 뒤 각 묶음에 같은 요약 함수를 적용하고 결과를 다시 하나의 표로 결합합니다.

$$\text{원자료 행} \rightarrow \text{조건별 묶음} \rightarrow \text{평균·중앙값·개수} \rightarrow \text{요약표}$$

평균만 제시하면 표본 수가 적은 집단과 많은 집단을 구분하기 어렵습니다. 따라서 `mean`, `median`, `count`를 함께 확인하면 중심과 표본 규모를 동시에 읽을 수 있습니다.

다음 실행 셀에서는 근무일 여부에 따라 시간별 대여건수를 묶고, 요약표와 평균 막대그래프를 연결합니다.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

bike = pd.DataFrame(
    {
        "day": ["Mon", "Mon", "Tue", "Tue", "Sat", "Sat", "Sun", "Sun"],
        "hour": [8, 17, 8, 17, 8, 17, 8, 17],
        "workingday": ["근무일"] * 4 + ["비근무일"] * 4,
        "rentals": [180, 260, 170, 245, 85, 135, 92, 150],
    }
)
grouped = (
    bike.groupby(["workingday", "hour"], as_index=False)
    .agg(mean_rentals=("rentals", "mean"), median_rentals=("rentals", "median"), n=("rentals", "size"))
)

print("[원자료] grain = 요일 × 시간")
display(bike)
print("\n[groupby 결과] grain = 근무일 여부 × 시간")
display(grouped)
print(f"행 수 변화: {len(bike)} → {len(grouped)}")

chart = grouped.pivot(index="hour", columns="workingday", values="mean_rentals").rename(
    columns={"근무일": "Working day", "비근무일": "Non-working day"}
)
ax = chart.plot.bar(figsize=(7.5, 3.5), color=["#72B7B2", "#4C78A8"])
ax.set(title="Mean rentals by hour and working-day status", xlabel="Hour", ylabel="Mean rentals")
ax.legend(title="Group")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## line plot은 시간의 순서와 변화 방향을 보존한다

선 그래프는 x축의 순서가 의미 있을 때 사용합니다. 연도, 월, 날짜, 시간처럼 앞뒤 관계가 있는 변수에 적합합니다.

선을 그리기 전에 다음을 확인합니다.

- 시간이 실제 순서대로 정렬됐는가?
- y축의 단위는 무엇인가?
- 선 하나는 어떤 집단을 뜻하는가?
- 점은 원자료인가, 평균이나 합계인가?
- 중간 시점이 비어 있다면 선으로 연결해도 되는가?

다음 실행 셀에서는 두 지역의 연도별 원자료를 그리고, 표의 행과 선의 점이 어떻게 연결되는지 확인합니다.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

emissions = pd.DataFrame(
    {
        "year": [2018, 2019, 2020, 2021, 2022] * 2,
        "region": ["A"] * 5 + ["B"] * 5,
        "co2_mt": [520, 510, 470, 495, 480, 310, 318, 292, 300, 288],
    }
).sort_values(["region", "year"])

print("[선의 각 점에 대응하는 원자료]")
display(emissions)
print("한 선 = 한 지역, 한 점 = 한 지역의 한 연도")

fig, ax = plt.subplots(figsize=(8, 3.6))
for region, part in emissions.groupby("region"):
    ax.plot(part["year"], part["co2_mt"], marker="o", linewidth=2, label=f"Region {region}")
ax.set(title="Annual CO₂ emissions by region", xlabel="Year", ylabel="CO₂ (Mt)")
ax.set_xticks(sorted(emissions["year"].unique()))
ax.legend()
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()


## bar plot과 box plot은 서로 다른 질문에 답한다

| 그림 | 주로 답하는 질문 | 잃기 쉬운 정보 |
|---|---|---|
| bar plot | 집단별 하나의 요약값은 얼마나 다른가? | 집단 내부의 분포와 표본 수 |
| box plot | 집단 내부의 중앙값·사분위 범위·극단값은 어떠한가? | 개별 관측의 시간 순서 |

같은 자료에서도 평균 막대만 보면 집단의 중심 차이가 강조되고, 상자그림을 보면 분포의 겹침과 극단값이 드러납니다. 다음 실행 셀에서는 두 그림을 나란히 놓고 결론이 어떻게 달라지는지 확인합니다.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

air = pd.DataFrame(
    {
        "site_type": ["도로변"] * 8 + ["도시배경"] * 8,
        "pm25": [24, 25, 27, 28, 30, 31, 33, 54, 18, 20, 21, 22, 23, 24, 26, 40],
    }
)
summary = air.groupby("site_type")["pm25"].agg(["count", "mean", "median", "min", "max"]).round(1)

print("[집단별 요약표]")
display(summary)
print("막대는 평균 하나를, 상자그림은 집단 안의 분포를 보여줍니다.")

roadside = air.loc[air["site_type"].eq("도로변"), "pm25"]
q1, median, q3 = roadside.quantile([0.25, 0.50, 0.75])
iqr = q3 - q1
lower_limit = q1 - 1.5 * iqr
upper_limit = q3 + 1.5 * iqr
lower_whisker = roadside.loc[roadside.ge(lower_limit)].min()
upper_whisker = roadside.loc[roadside.le(upper_limit)].max()
outliers = roadside.loc[roadside.lt(lower_limit) | roadside.gt(upper_limit)].tolist()
anatomy = pd.DataFrame(
    {
        "상자그림 요소": ["lower whisker", "Q1", "median", "Q3", "upper whisker", "IQR", "outlier"],
        "도로변 PM2.5": [lower_whisker, q1, median, q3, upper_whisker, iqr, outliers],
        "읽는 의미": [
            "1.5×IQR 범위 안의 가장 작은 관측",
            "관측값의 25%가 이 값 이하",
            "관측값의 50%가 이 값 이하",
            "관측값의 75%가 이 값 이하",
            "1.5×IQR 범위 안의 가장 큰 관측",
            "Q3 − Q1: 가운데 50%의 폭",
            "두 수염 바깥의 관측",
        ],
    }
)
print("[도로변 상자그림을 구성하는 값]")
display(anatomy)

order = ["도로변", "도시배경"]
plot_order = ["Roadside", "Urban background"]
plot_air = air.assign(
    plot_group=air["site_type"].map({"도로변": "Roadside", "도시배경": "Urban background"})
)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.7))
summary.loc[order, "mean"].plot.bar(ax=axes[0], color=["#E45756", "#4C78A8"])
axes[0].set_xticklabels(plot_order)
axes[0].set(title="One summary per group", xlabel="", ylabel="Mean PM2.5")
axes[0].tick_params(axis="x", rotation=0)
sns.boxplot(
    data=plot_air, x="plot_group", y="pm25", order=plot_order,
    color="#72B7B2", width=0.55, showfliers=False, ax=axes[1]
)
sns.stripplot(
    data=plot_air, x="plot_group", y="pm25", order=plot_order,
    color="#1F2937", size=5, jitter=0.06, alpha=0.65, ax=axes[1]
)
axes[1].annotate("lower whisker", xy=(0, lower_whisker), xytext=(0.30, lower_whisker - 3), arrowprops={"arrowstyle": "->"})
axes[1].annotate("median", xy=(0, median), xytext=(0.30, median + 1), arrowprops={"arrowstyle": "->"})
axes[1].annotate("Q1", xy=(0.12, q1), xytext=(0.42, q1 - 3), arrowprops={"arrowstyle": "->"})
axes[1].annotate("Q3", xy=(0.12, q3), xytext=(0.42, q3 + 3), arrowprops={"arrowstyle": "->"})
axes[1].annotate("upper whisker", xy=(0, upper_whisker), xytext=(0.33, upper_whisker + 3), arrowprops={"arrowstyle": "->"})
axes[1].annotate("outlier", xy=(0, outliers[0]), xytext=(0.38, outliers[0] - 1), arrowprops={"arrowstyle": "->"})
axes[1].set(title="Distribution inside each group", xlabel="", ylabel="PM2.5")
plt.tight_layout()
plt.show()


## 그림의 제목과 축에 분석 조건을 남기기

연구용 그림은 색과 모양보다 **무엇을 어떤 조건으로 계산했는지**가 먼저 드러나야 합니다.

- 제목: 비교 대상과 기간
- x축: 시간 또는 집단의 의미
- y축: 변수명과 단위
- 범례: 색·선·기호가 뜻하는 집단
- 주석 또는 캡션: 평균·합계 등 집계함수와 표본 범위

관찰한 패턴은 “함께 나타났다”라고 표현할 수 있지만, 그림만으로 원인을 증명했다고 말하지 않습니다.


## 이번 교시의 재현 절차

1. 원자료의 grain과 key를 확인합니다.
2. `long → wide → long` 변환 전후의 key별 값을 비교합니다.
3. `groupby`로 집계한 뒤 새 grain과 표본 수를 기록합니다.
4. 시간 순서는 line plot, 집단 요약은 bar plot, 분포는 box plot으로 확인합니다.
5. 표와 그림이 답하는 질문, 답하지 못하는 질문을 한 문장씩 적습니다.

핵심 산출물은 장식적인 그림이 아니라 **변환·집계·시각화의 각 단계에서 관측의 의미가 추적되는 분석 기록**입니다.


## 실행 환경과 입력자료

| 재현 조건 | 확인할 출력 |
|---|---|
| 입력자료 | 실제 파일 경로와 파일명 |
| 자료 구조 | 데이터 소개에 제시한 예상 `shape`와 필수 열 |
| 실행환경 | Python과 주요 라이브러리 버전 |

다음 셀은 분석에 사용한 파일과 실행환경을 기록합니다. 파일 경로 또는 `shape`가 강의 기준과 다르면 입력 파일과 버전을 확인한 뒤 분석을 진행합니다.

> **실행 전 확인:** 이번 분석의 입력 파일과 예상 행 수는 무엇인가요?


In [ ]:
from pathlib import Path
import platform
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 130)
warnings.filterwarnings("ignore", category=FutureWarning)

def find_course_data(relative_path: str) -> Path:
    """0교시에서 준비한 data 폴더의 파일을 찾습니다."""
    relative = Path(relative_path)
    checked = []
    for start in [Path.cwd(), *Path.cwd().parents, Path("/content")]:
        candidate = (start / "data" / relative).resolve()
        if candidate in checked:
            continue
        checked.append(candidate)
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"데이터 파일을 찾지 못했습니다: {relative_path}. "
        "과정 폴더의 00_setup.ipynb를 먼저 실행하십시오."
    )
print("Python:", platform.python_version())
print("pandas:", pd.__version__)


## `groupby` 결과가 만들어지는 과정

각 행을 근무일 여부로 나누고 평균·중앙값·관측 수를 계산합니다. 집계 전후에 한 행의 의미가 어떻게 달라지는지 확인하세요.


In [ ]:
group_example = pd.DataFrame({
    "workingday": [0, 0, 0, 1, 1, 1],
    "rentals": [10, 12, 50, 30, 32, 34],
})
group_summary = group_example.groupby("workingday", as_index=False).agg(
    mean_rentals=("rentals", "mean"),
    median_rentals=("rentals", "median"),
    observations=("rentals", "size"),
)
print("[집계 전: 한 행은 한 관측]")
display(group_example)
print("[집계 후: 한 행은 근무일 유형 한 그룹의 요약]")
display(group_summary.round(2))
print("비근무일 그룹은 값 50 때문에 평균과 중앙값의 차이가 큽니다.")


## 1. 변환 전 데이터 구조

표의 형식을 바꾸기 전에 원자료의 구조를 기록합니다.

| 항목 | CO₂ 원자료 |
|---|---|
| grain | 지역 × 연도 |
| key 후보 | `code + year` |
| 측정값 | `emissions_total` |
| 단위 | tonnes CO₂ |
| 예상 key 중복 | 0건 |

코드 셀은 첫 행과 `code + year` 중복 수를 출력합니다. 이 기록이 `pivot`, `melt`, `groupby` 이후의 비교 기준입니다.

> **확인 질문:** `year`만으로 원자료의 한 행을 식별할 수 없는 이유는 무엇입니까?


In [ ]:
CO2_PATH = find_course_data("co2/owid_co2_sample.csv")
co2 = pd.read_csv(CO2_PATH)
analysis_name = "OWID CO2 country-year snapshot"

print("shape:", co2.shape)
print("duplicate code-year keys:", co2.duplicated(["code", "year"]).sum())
display(co2.head(3))


## 2. Long–Wide 구조 변환

`pivot`과 `melt`는 같은 측정값을 다른 행·열 구조에 배치합니다.

| 단계 | 한 행 | 국가 정보 | 배출량 |
|---|---|---|---|
| long | 국가 × 연도 | `code` 열의 값 | `emissions_total` 열 |
| wide | 연도 | 국가별 열 이름 | 국가별 열의 값 |
| long 복원 | 국가 × 연도 | `code` 열의 값 | 단일 측정 열 |

구조 변환 과정에서는 평균이나 합계를 계산하지 않습니다. 검증 기준은 `code + year`별 배출량의 보존입니다.

> **출력 예측:** wide 표에서 `code` 값은 어느 위치로 이동합니까?


In [ ]:
country_only = co2.loc[co2["code"].isin(["KOR", "JPN", "CHN", "USA"])].copy()

wide = country_only.pivot(index="year", columns="code", values="emissions_total")
long_again = (
    wide.reset_index()
        .melt(id_vars="year", var_name="code", value_name="emissions_total")
        .sort_values(["code", "year"])
        .reset_index(drop=True)
)

print("country-only long shape:", country_only.shape)
print("wide shape:", wide.shape)
print("long-again shape:", long_again.shape)
display(wide.head(3))


### 출력 해석: long → wide → long

같은 연도의 값이 행에 반복되는지, 국가별 열로 펼쳐지는지, 다시 행으로 모이는지 표 세 개를 순서대로 비교합니다.


In [ ]:
sample_year = int(country_only["year"].min())

print(f"1) LONG 원자료 — {sample_year}년, 국가가 여러 행에 있습니다.")
display(country_only.loc[country_only["year"].eq(sample_year), ["year", "code", "emissions_total"]])

print("2) WIDE 변환 — 같은 연도가 한 행이고 국가가 열이 됩니다.")
display(wide.loc[[sample_year]].reset_index())

print("3) LONG 복원 — 국가가 다시 code 열의 값으로 내려옵니다.")
display(long_again.loc[long_again["year"].eq(sample_year)].sort_values("code"))

print("shape 변화:", country_only.shape, "→", wide.shape, "→", long_again.shape)


## 3. 집계 후 관측 단위

`groupby().agg()`는 여러 관측으로 요약값을 계산합니다.

| 구분 | 집계 전 | 집계 후 |
|---|---|---|
| 한 행 | 지역 × 연도 | 지역 |
| 값 | 연도별 배출량 | 기간 평균 배출량 |
| key | `code + year` | `code` |

집계표에서 표본 수와 집계함수를 확인한 뒤 시각화를 작성합니다. 그림의 제목과 축에는 변수, 단위, 기간, 집계함수를 명시합니다.

> **확인 질문:** 국가별 평균 막대 하나는 몇 개 연도 관측을 요약합니까?


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plot_data = country_only.assign(emissions_gt=country_only["emissions_total"] / 1_000_000_000)
result_table = (
    plot_data.groupby(["entity", "code"], as_index=False)
             .agg(mean_gt=("emissions_gt", "mean"),
                  min_year=("year", "min"), max_year=("year", "max"), rows=("year", "size"))
             .sort_values("mean_gt", ascending=False)
)
display(result_table)

fig, ax = plt.subplots(figsize=(9, 4.6))
sns.lineplot(data=plot_data, x="year", y="emissions_gt", hue="entity", linewidth=2, ax=ax)
ax.set(title="Annual territorial CO2 emissions", xlabel="Year", ylabel="Gt CO2/year")
ax.legend(title="Region", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


### 출력 해석: 원자료와 집계표의 grain

특정 국가의 여러 연도 행이 집계 뒤 한 행으로 줄어드는 과정을 확인합니다.


In [ ]:
sample_code = "KOR"
print("집계 전: 국가 × 연도 grain")
display(plot_data.loc[plot_data["code"].eq(sample_code), ["entity", "year", "emissions_gt"]].head(5))
print("집계 후: 국가 grain")
display(result_table.loc[result_table["code"].eq(sample_code)])


## 4. 구조 변환의 값 보존 검증

Long–wide–long 왕복 변환은 다음 조건으로 검증합니다.

1. 원본 long 표와 복원한 long 표를 `code + year`로 정렬합니다.
2. 두 표의 key 집합을 비교합니다.
3. 각 key에 연결된 `emissions_total` 값을 비교합니다.
4. 행 순서의 차이와 관측값의 차이를 구분합니다.

검증 기준은 같은 key에 같은 배출량이 연결되어 있는가입니다. 이 검사는 구조 변환의 값 보존을 평가하며, 집계함수 선택의 타당성은 별도 항목입니다.

> **확인 질문:** 행 순서가 달라도 같은 관측 집합으로 판단할 수 있는 조건은 무엇입니까?


In [ ]:
original_check = (
    country_only[["code", "year", "emissions_total"]]
    .sort_values(["code", "year"])
    .reset_index(drop=True)
)
roundtrip_check = long_again[["code", "year", "emissions_total"]]

checks = pd.Series({
    "unique source key": country_only.duplicated(["code", "year"]).sum() == 0,
    "roundtrip row count": len(original_check) == len(roundtrip_check),
    "roundtrip values retained": original_check.equals(roundtrip_check),
    "four complete series": result_table["rows"].eq(35).all(),
}, name="passed")
display(checks.to_frame())
assert checks.all()
print("구조 변환 검증: PASS")


## ChatGPT 저장 응답 검증: 표 변환과 그림의 질문 맞추기

이번 요청은 long·wide 중 하나를 정답으로 고르게 하는 것이 아닙니다. **변환 전후의 식별자와 값이 보존되는지**, 그리고 선택한 집계와 그림이 연구질문에 맞는지를 점검합니다.

답변의 변환 절차는 실제 `shape`, key 집합과 값 비교로 확인하고, 그래프 제안은 축·단위·집계함수가 명시된 경우에만 사용합니다.


In [ ]:
review_prompt = f"""
다음 공개자료의 구조 변환·집계·선그래프를 검토해 주세요.

- 분석 이름: {analysis_name}
- 원자료 관측 단위: 국가 한 곳의 한 연도
- 원자료 일부:
{country_only[["entity", "code", "year", "emissions_total"]].head(8).to_string(index=False)}
- wide 표 일부:
{wide.head(5).to_string()}
- long으로 되돌린 표 일부:
{long_again.head(8).to_string(index=False)}
- 국가별 집계표:
{result_table.to_string(index=False)}
- Notebook 검증 결과:
{checks.to_string()}

아래 순서로 답해 주세요.
1. 원자료·wide 표·집계표에서 한 행이 각각 무엇을 뜻하는지
2. long → wide → long 변환에서 확인해야 할 key·행 수·값 보존 조건
3. 국가별 평균을 계산한 집계함수가 답하는 질문과 답하지 못하는 질문
4. 선그래프의 x축·y축·색·단위·기간을 포함한 해석문장 한 개

제약: 제공하지 않은 heatmap이나 원인을 언급하지 말고, 검증 결과가 PASS인지
실제 표에서 확인해 주세요. 표에 없는 수치를 만들지 마세요.
""".strip()
print("[ChatGPT에 복사할 프롬프트]\n")
print(review_prompt)


### ChatGPT 저장 응답을 검증할 때

1. 변환 전후 key 집합과 값 보존 검사를 먼저 실행합니다.
2. 집계 뒤 한 행의 의미와 집계함수를 한 문장으로 적습니다.
3. 그래프의 축·단위·표본 범위가 빠진 제안은 보완한 뒤 사용합니다.


## 해석 범위

집계표와 시각화는 선택한 기간과 조건에서 나타난 자료의 패턴을 요약합니다.

날씨·계절·연도 구성의 차이를 통제한 결과는 아니므로, 시간대나 근무일이 수요를 변화시켰다는 인과적 결론으로 해석하지 않습니다. 적용 데이터가 달라지면 집계함수, 표본범위와 관측 단위를 다시 명시합니다.


## 오늘의 결론과 다음 단계

### 이번 교시의 결론

연구질문에 맞춰 한 행의 의미를 정하고, 집계와 구조 변환에서 key와 값이 보존되는지 확인한 뒤 그림을 선택합니다.

### 적용 순서

1. `02_tidy_data_visualization_follow.ipynb`에서 핵심 시연을 같은 조건으로 재현합니다.
2. `02_tidy_data_visualization_practice.ipynb`에서 지정된 데이터 또는 조건 하나만 바꿔 비교합니다.
3. 변경조건, 비교표 또는 그림, 결과문장과 현재 검증하지 못한 범위를 함께 기록합니다.

### 실습 중 확인할 질문

- `groupby`의 분할–적용–결합 각 단계에서 행은 어떻게 나뉘고 요약되며 다시 합쳐집니까?
- 같은 그룹에 평균, 중앙값, 합계를 적용하면 각각 어떤 연구 질문에 답하게 됩니까?
- 시간별 변화와 그룹별 분포를 보여줄 때 각각 어떤 그림을 선택하고, 축·단위·집계함수를 어떻게 밝히겠습니까?

마지막 `PASS`는 Notebook에 적힌 실행조건을 확인한 결과이며 연구결론의 타당성을 대신하지 않습니다.
